# Olist E-commerce Analysis with PySpark

Customer geography, geolocation quality, purchase amounts, and delivery experience.

**Author:** Farida Belal  
**Origin:** Adapted from CSCI 443 Homework 6 at the University of Mississippi. The analysis questions originated in coursework; this portfolio edition reorganizes the submitted work and revises documentation and selected calculations.

## Questions

1. How are customer records distributed across states and regions?
2. Can ZIP-prefix median coordinates reduce the impact of geographic outliers?
3. Do mean order payment amounts differ between São Paulo and Rio de Janeiro **states**?
4. How is delivery time associated with review score?

## Run status and setup

This revised notebook has been checked for Python syntax and notebook structure, but has **not been rerun against the Olist CSV files**. Outputs were cleared to avoid presenting old results as outputs of revised code. Historical findings below come from saved outputs in the submitted homework.

Use a Databricks Python notebook with Spark, a readable Unity Catalog volume, and NumPy, pandas, SciPy, Matplotlib, and psutil available. Download and extract the [Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), upload the five CSVs listed in the README, update `DATA_DIR`, then run all cells in order. This notebook uses Databricks `display` and `dbutils`.

## Historical findings — original saved run

- 99,441 distinct customer IDs and 96,096 distinct customer-unique IDs.
- 1,000,163 geolocation records covering 19,015 ZIP prefixes.
- 40 geolocation rows outside the chosen bounding box with a one-degree margin; seven ZIP prefixes had median coordinates outside the unexpanded box.
- 576 of 1,000,134 retained geolocation rows were more than 200 km from their ZIP-prefix median (about 0.0576%). These are row counts, not counts of distinct ZIP prefixes.
- Mean summed payment per order: São Paulo 143.69 and Rio de Janeiro 166.85; SP − RJ = −23.16. Original Welch t-test: t ≈ −9.83, p ≈ 9.81 × 10⁻²³.
- Original delivery-time/review-score correlation: r ≈ −0.334.

The revised CSV reader handles quoted multiline records. That correction can change review parsing and downstream results; rerun before treating these historical findings as current results.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

spark = SparkSession.builder.appName("OlistEcommerceAnalysis").getOrCreate()
DATA_DIR = "/Volumes/workspace/default/files/olist"  # Change to your volume.
SEED = 42
N_PERMUTATIONS = 5000

def read_olist(filename):
    # Olist review comments can contain quoted commas, quotes, and newlines.
    return (spark.read.option("header", True)
            .option("inferSchema", True)
            .option("multiLine", True)
            .option("quote", '"').option("escape", '"')
            .csv(f"{DATA_DIR.rstrip('/')}/{filename}"))


## 1. Data access

Inspect the volume and load customer records.

In [ ]:
display(dbutils.fs.ls(DATA_DIR))

In [ ]:
df_customers = read_olist("olist_customers_dataset.csv")

display(df_customers.limit(10))

### File sizes

Disk size is a first sizing check; in-memory DataFrames and temporary conversions can use more memory.

In [ ]:
files = dbutils.fs.ls(DATA_DIR)

for f in files:
    print(f.name, f.size)


In [ ]:
total_size = 0

for f in files:
    size_mb = f.size / (1024 * 1024)
    total_size += size_mb
    print(f"{f.name}: {size_mb:.2f} MB")

print(f"\nTotal size: {total_size:.2f} MB")

### Customer identifiers

`customer_id` links a customer record to an order. `customer_unique_id` identifies returning customers across orders. State and region charts below count records, not distinct people.

In [ ]:
df_customers.selectExpr(
    "count(distinct customer_id) as unique_customer_id",
    "count(distinct customer_unique_id) as unique_customer_unique_id"
).show()

### Missing-value screening

Check nulls, empty strings, and placeholder strings. Zero is checked only for ZIP prefixes; zero latitude or longitude is not automatically missing. This is a screening rule, not proof of complete data quality.

In [ ]:
from functools import reduce
from pyspark.sql.functions import lower, trim

def find_missing_rows(df):
    conditions = []
    for field in df.schema.fields:
        name = field.name
        cond = col(name).isNull()
        if name.endswith("zip_code_prefix"):
            cond = cond | (col(name) == 0)
        if field.dataType.simpleString() == "string":
            cond = cond | lower(trim(col(name))).isin("", "na", "n/a", "none")
        conditions.append(cond)
    return df.filter(reduce(lambda a, b: a | b, conditions))


In [ ]:
missing_df = find_missing_rows(df_customers)

display(missing_df.limit(10))
print("Number of rows with missing/invalid data:", missing_df.count())

### State coverage

In [ ]:
print("Number of unique states:", df_customers.select("customer_state").distinct().count())
display(df_customers.select("customer_state").distinct().orderBy("customer_state"))

### Customer records by state

Aggregate with Spark before collecting the small result for plotting.

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# Count customer records per state in Spark
state_counts = (
    df_customers
    .groupBy("customer_state")
    .count()
    .orderBy("customer_state")
)

display(state_counts)

In [ ]:
state_counts_pd = state_counts.toPandas()

plt.figure(figsize=(12, 6))
plt.bar(state_counts_pd["customer_state"], state_counts_pd["count"])
plt.xlabel("State")
plt.ylabel("Number of Customer Records")
plt.title("Number of Customer Records by Brazilian State")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Customer records by region

Create a derived DataFrame with region labels; the original Spark DataFrame is unchanged.

In [ ]:
from pyspark.sql.functions import col, when

df_customers_region = (
    df_customers
    .withColumn(
        "region",
        when(col("customer_state").isin("ES", "MG", "RJ", "SP"), "Southeast")
        .when(col("customer_state").isin("PR", "SC", "RS"), "South")
        .when(col("customer_state").isin("AC", "AP", "AM", "PA", "RO", "RR", "TO"), "North")
        .when(col("customer_state").isin("AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"), "Northeast")
        .when(col("customer_state").isin("GO", "MT", "MS", "DF"), "Central-West")
        .otherwise("Unknown")
    )
)

display(df_customers_region.limit(10))

In [ ]:
print("Unclassified customer records:", df_customers_region.filter(col("region") == "Unknown").count())


In [ ]:
region_counts = (
    df_customers_region
    .groupBy("region")
    .count()
    .orderBy("region")
)

display(region_counts)

In [ ]:
region_counts_pd = region_counts.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(region_counts_pd["region"], region_counts_pd["count"])
plt.xlabel("Region")
plt.ylabel("Number of Customer Records")
plt.title("Number of Customer Records by Region")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 2. Geolocation quality

Multiple coordinate records can share the same ZIP prefix. Retain raw observations and create separate derived views.

In [ ]:
geo_df = read_olist("olist_geolocation_dataset.csv")

display(geo_df.limit(10))

### Row counts and ZIP-prefix coverage

In [ ]:
total_rows = geo_df.count()
print("Total rows:", total_rows)

In [ ]:
distinct_zips = geo_df.select("geolocation_zip_code_prefix").distinct().count()
print("Distinct zip prefixes:", distinct_zips)

### Missing-value screening

In [ ]:
geo_missing_df = find_missing_rows(geo_df)

display(geo_missing_df.limit(10))
print("Number of rows with missing or suspicious data:", geo_missing_df.count())

### Estimate driver memory use

A small sample provides a rough per-row estimate. It excludes conversion overhead and is not a guarantee that collecting the full dataset is safe. Geographic plotting below uses this sample.

In [ ]:
import psutil

# 1) take a small sample from the Spark DataFrame
sample_fraction = 0.01   # 1%
geo_sample = geo_df.sample(withReplacement=False, fraction=sample_fraction, seed=42)

# 2) convert sample to pandas
geo_sample_pd = geo_sample.toPandas()

# 3) measure sample memory usage
sample_memory_bytes = geo_sample_pd.memory_usage(deep=True).sum()
sample_memory_mb = sample_memory_bytes / (1024**2)

# 4) estimate full dataset memory from the sample
sample_rows = len(geo_sample_pd)
full_rows = geo_df.count()
if sample_rows == 0:
    raise ValueError("Empty memory-estimation sample; increase sample_fraction.")
estimated_full_bytes = sample_memory_bytes / sample_rows * full_rows
estimated_full_mb = estimated_full_bytes / (1024**2)
estimated_full_gb = estimated_full_bytes / (1024**3)

# also helpful: compare row counts
sample_rows = len(geo_sample_pd)
full_rows = geo_df.count()

# 5) inspect driver memory
vm = psutil.virtual_memory()
total_mem_gb = vm.total / (1024**3)
available_mem_gb = vm.available / (1024**3)

print(f"Sample fraction: {sample_fraction}")
print(f"Sample rows: {sample_rows}")
print(f"Full rows: {full_rows}")
print(f"Sample memory usage: {sample_memory_mb:.2f} MB")
print(f"Estimated full dataset memory: {estimated_full_mb:.2f} MB ({estimated_full_gb:.2f} GB)")
print(f"Driver total memory: {total_mem_gb:.2f} GB")
print(f"Driver available memory: {available_mem_gb:.2f} GB")

### Geographic screening

Use an approximate rectangular boundary from the coursework, with a one-degree plotting margin. This does not establish whether a point lies inside the national border and can flag legitimate locations outside the chosen rectangle. Treat flags as candidates for review.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Reuse the small seeded sample for plotting.
geo_pd = geo_sample_pd

# Brazil bounding box in decimal degrees
brazil_min_lat = -33.7508
brazil_max_lat = 5.2694
brazil_min_lng = -73.9831
brazil_max_lng = -34.7931

# 1-degree margin
margin = 1.0
plot_min_lat = brazil_min_lat - margin
plot_max_lat = brazil_max_lat + margin
plot_min_lng = brazil_min_lng - margin
plot_max_lng = brazil_max_lng + margin

plt.figure(figsize=(10, 10))
plt.scatter(
    geo_pd["geolocation_lng"],
    geo_pd["geolocation_lat"],
    s=2,
    alpha=0.3
)

ax = plt.gca()
rect = patches.Rectangle(
    (plot_min_lng, plot_min_lat),
    plot_max_lng - plot_min_lng,
    plot_max_lat - plot_min_lat,
    linewidth=2,
    fill=False
)
ax.add_patch(rect)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Sampled Olist Geolocations: Approximate Bounding Box")
plt.xlim(plot_min_lng - 1, plot_max_lng + 1)
plt.ylim(plot_min_lat - 1, plot_max_lat + 1)
plt.show()

### Rows outside the expanded rectangle

In [ ]:
from pyspark.sql.functions import col

outside_margin_df = geo_df.filter(
    (col("geolocation_lat") < plot_min_lat) |
    (col("geolocation_lat") > plot_max_lat) |
    (col("geolocation_lng") < plot_min_lng) |
    (col("geolocation_lng") > plot_max_lng)
)

outside_margin_count = outside_margin_df.count()
print("Rows outside the 1-degree-margin bounding box:", outside_margin_count)

display(outside_margin_df.limit(20))

### Distinct city/state labels

Spelling variants can create multiple labels for one city, so this is not an authoritative count of cities.

In [ ]:
city_state_df = geo_df.select("geolocation_city", "geolocation_state").distinct()

num_city_state_pairs = city_state_df.count()
print("Number of distinct city-state pairs:", num_city_state_pairs)

display(city_state_df.limit(20))

### Median coordinates by ZIP prefix

Use Spark approximate medians to reduce the effect of extreme coordinates. Median latitude and longitude are computed separately and need not identify an actual address.

In [ ]:
#step 1
from pyspark.sql import functions as F

geo_df_median = (
    geo_df
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.expr("percentile_approx(geolocation_lat, 0.5)").alias("median_lat"),
        F.expr("percentile_approx(geolocation_lng, 0.5)").alias("median_lng"),
        F.count("*").alias("row_count")
    )
)

display(geo_df_median.limit(10))

In [ ]:
#step 2

geo_median_pd = geo_df_median.toPandas()

plt.figure(figsize=(10, 10))
plt.scatter(
    geo_median_pd["median_lng"],
    geo_median_pd["median_lat"],
    s=4,
    alpha=0.5
)

ax = plt.gca()
rect = patches.Rectangle(
    (plot_min_lng, plot_min_lat),
    plot_max_lng - plot_min_lng,
    plot_max_lat - plot_min_lat,
    linewidth=2,
    fill=False
)
ax.add_patch(rect)

plt.xlabel("Median Longitude")
plt.ylabel("Median Latitude")
plt.title("Median Geolocation by Zip Prefix")
plt.xlim(plot_min_lng - 1, plot_max_lng + 1)
plt.ylim(plot_min_lat - 1, plot_max_lat + 1)
plt.show()


### Inspect ZIP prefixes with medians outside the rectangle

In [ ]:
median_outside_df = geo_df_median.filter(
    (col("median_lat") < brazil_min_lat) |
    (col("median_lat") > brazil_max_lat) |
    (col("median_lng") < brazil_min_lng) |
    (col("median_lng") > brazil_max_lng)
)

print("Zip prefixes whose median falls outside the chosen rectangle:", median_outside_df.count())
display(median_outside_df)

In [ ]:
#step 4

bad_zip_df = geo_df.join(
    median_outside_df.select("geolocation_zip_code_prefix"),
    on="geolocation_zip_code_prefix",
    how="inner"
)

display(bad_zip_df.orderBy("geolocation_zip_code_prefix"))

In [ ]:
#step 5
bad_city_counts = (
    bad_zip_df
    .select("geolocation_city", "geolocation_state", "geolocation_zip_code_prefix")
    .distinct()
    .groupBy("geolocation_city", "geolocation_state")
    .count()
    .orderBy(F.desc("count"))
)

display(bad_city_counts)

In [ ]:
single_row_bad_count = median_outside_df.filter(col("row_count") == 1).count()
total_bad_count = median_outside_df.count()

print("Bad median zip prefixes with only one row:", single_row_bad_count)
print("Total bad median zip prefixes:", total_bad_count)



### Distance from ZIP-prefix medians

Keep a derived view whose median coordinates pass the rectangular screen. Compute great-circle distance with the haversine formula. These filters are exploratory assumptions, not verified address corrections.

In [ ]:
geo_df_median_valid = geo_df_median.filter(
    (col("median_lat") >= brazil_min_lat) &
    (col("median_lat") <= brazil_max_lat) &
    (col("median_lng") >= brazil_min_lng) &
    (col("median_lng") <= brazil_max_lng)
)

In [ ]:
geo_df_both = geo_df.join(
    geo_df_median_valid.select("geolocation_zip_code_prefix", "median_lat", "median_lng"),
    on="geolocation_zip_code_prefix",
    how="inner"
)

display(geo_df_both.limit(10))

In [ ]:
from pyspark.sql.functions import radians, sin, cos, asin, sqrt, lit

R = 6371.0  # Earth radius in km

geo_df_both = (
    geo_df_both
    .withColumn("lat1", radians(col("geolocation_lat")))
    .withColumn("lng1", radians(col("geolocation_lng")))
    .withColumn("lat2", radians(col("median_lat")))
    .withColumn("lng2", radians(col("median_lng")))
    .withColumn("dlat", col("lat2") - col("lat1"))
    .withColumn("dlng", col("lng2") - col("lng1"))
    .withColumn(
        "a",
        sin(col("dlat") / 2) ** 2 +
        cos(col("lat1")) * cos(col("lat2")) * sin(col("dlng") / 2) ** 2
    )
    .withColumn("distance_km", lit(2 * R) * asin(sqrt(F.least(lit(1.0), F.greatest(lit(0.0), col("a"))))))
)

In [ ]:
geo_far_df = geo_df_both.filter(col("distance_km") > 200)

far_count = geo_far_df.count()
total_count = geo_df_both.count()
far_fraction = far_count / total_count if total_count else float("nan")

print("Rows more than 200 km from median:", far_count)
print("Total rows considered:", total_count)
print("Fraction more than 200 km from median:", far_fraction)

display(geo_far_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_city",
    "geolocation_state",
    "geolocation_lat",
    "geolocation_lng",
    "median_lat",
    "median_lng",
    "distance_km"
).orderBy(F.desc("distance_km")))

In [ ]:
median_pd = geo_df_median_valid.toPandas()
far_pd = geo_far_df.toPandas()

plt.figure(figsize=(10, 10))

plt.scatter(
    median_pd["median_lng"],
    median_pd["median_lat"],
    s=4,
    alpha=0.5,
    label="Median coordinates"
)

plt.scatter(
    far_pd["geolocation_lng"],
    far_pd["geolocation_lat"],
    s=8,
    alpha=0.7,
    label="Original points > 200 km from median"
)

ax = plt.gca()
rect = patches.Rectangle(
    (brazil_min_lng, brazil_min_lat),
    brazil_max_lng - brazil_min_lng,
    brazil_max_lat - brazil_min_lat,
    linewidth=2,
    fill=False
)
ax.add_patch(rect)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Median Coordinates and Original Points More Than 200 km Away")
plt.legend()
plt.show()

## 3. Order payments: São Paulo versus Rio de Janeiro

Sum payment records per order before joining customers. Compare **states**, not cities. All order statuses are retained; summed payment value is not net revenue or profit. Repeated buyers can introduce dependence across orders.

Welch test: H₀: equal state mean payment amounts; H₁: different means. The test allows unequal variances but assumes independent observations.

In [ ]:
from pyspark.sql import functions as F

# load needed tables if not already loaded
orders_df = read_olist("olist_orders_dataset.csv")
payments_df = read_olist("olist_order_payments_dataset.csv")
customers_df = read_olist("olist_customers_dataset.csv")

# one total purchase amount per order
order_payments_df = (
    payments_df
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("total_purchase_amount"))
)

# join to customers through orders
purchase_state_df = (
    orders_df
    .join(customers_df, on="customer_id", how="inner")
    .join(order_payments_df, on="order_id", how="inner")
    .select("order_id", "customer_state", "total_purchase_amount")
)

display(purchase_state_df.limit(10))

In [ ]:
sp_rj_df = purchase_state_df.filter(F.col("customer_state").isin("SP", "RJ"))

sp_rj_pd = sp_rj_df.toPandas()

sp_amounts = sp_rj_pd.loc[sp_rj_pd["customer_state"] == "SP", "total_purchase_amount"]
rj_amounts = sp_rj_pd.loc[sp_rj_pd["customer_state"] == "RJ", "total_purchase_amount"]

print("SP orders:", len(sp_amounts))
print("RJ orders:", len(rj_amounts))
print("SP mean:", sp_amounts.mean())
print("RJ mean:", rj_amounts.mean())

In [ ]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(sp_amounts, rj_amounts, equal_var=False)

print("Welch t-test statistic:", t_stat)
print("p-value:", p_value)
print("Observed mean difference (SP - RJ):", sp_amounts.mean() - rj_amounts.mean())

### Permutation comparison

Shuffle state labels under an exchangeability null. This is stronger than equality of means alone; differing distributions or repeated customers can invalidate that interpretation. Count absolute differences at least as large as observed and use `(extreme + 1) / (B + 1)`. With 5,000 draws, the minimum reported estimate is 1/5,001, approximately 0.0002, not zero. See [SciPy’s randomized-test convention](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sp_vals = sp_amounts.to_numpy()
rj_vals = rj_amounts.to_numpy()

observed_diff = sp_vals.mean() - rj_vals.mean()

combined = np.concatenate([sp_vals, rj_vals])
n_sp = len(sp_vals)
n_perm = N_PERMUTATIONS

perm_diffs = np.empty(n_perm)

rng = np.random.default_rng(SEED)

for i in range(n_perm):
    permuted = rng.permutation(combined)
    perm_sp = permuted[:n_sp]
    perm_rj = permuted[n_sp:]
    perm_diffs[i] = perm_sp.mean() - perm_rj.mean()

# two-sided p-value
extreme_count = np.count_nonzero(np.abs(perm_diffs) >= abs(observed_diff))
perm_p_value = (extreme_count + 1) / (n_perm + 1)

print("Observed difference (SP - RJ):", observed_diff)
print("Permutation p-value:", perm_p_value)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(perm_diffs, bins=40)
plt.axvline(observed_diff, linewidth=2, label="Observed difference")
plt.xlabel("Permuted mean differences (SP - RJ)")
plt.ylabel("Frequency")
plt.title("Permutation Test: Purchase Amount Difference")
plt.legend()
plt.show()

## 4. Delivery time and review score

Join reviews to orders with delivery dates and retain nonnegative delivery intervals. The unit is a joined review record; multiple reviews can give an order extra weight. Review scores are ordinal, and Pearson correlation is a descriptive linear summary. Association does not establish that delivery time causes the score.

In [ ]:
reviews_df = read_olist("olist_order_reviews_dataset.csv")

delivery_review_df = (
    orders_df
    .join(reviews_df, on="order_id", how="inner")
    .filter(
        F.col("order_purchase_timestamp").isNotNull() &
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("review_score").isNotNull()
    )
    .withColumn(
        "delivery_time_days",
        (
            F.unix_timestamp("order_delivered_customer_date") -
            F.unix_timestamp("order_purchase_timestamp")
        ) / 86400.0
    )
    .select("order_id", "delivery_time_days", "review_score")
    .filter(F.col("delivery_time_days") >= 0)
)

display(delivery_review_df.limit(10))

In [ ]:
from pyspark.sql.functions import col

delivery_review_df = delivery_review_df.withColumn(
    "delivery_time_days", col("delivery_time_days").cast("double")
).withColumn(
    "review_score", col("review_score").cast("double")
)

In [ ]:
corr_value = delivery_review_df.stat.corr("delivery_time_days", "review_score")
print("Pearson correlation:", corr_value)

### Permutation assessment of negative association

Shuffle review scores across the fixed delivery times. The one-sided alternative is negative association. Exchangeability and independent records are assumptions; multiple reviews and repeated customers are limitations. This is exploratory evidence, not a causal test.

In [ ]:
delivery_review_pd = delivery_review_df.select("delivery_time_days", "review_score").toPandas().dropna()
x = delivery_review_pd["delivery_time_days"].to_numpy()
y = delivery_review_pd["review_score"].to_numpy()

observed_r = np.corrcoef(x, y)[0, 1]

n_perm = N_PERMUTATIONS
perm_rs = np.empty(n_perm)

rng = np.random.default_rng(SEED)

for i in range(n_perm):
    perm_y = rng.permutation(y)
    perm_rs[i] = np.corrcoef(x, perm_y)[0, 1]

# one-sided p-value for negative correlation
extreme_count = np.count_nonzero(perm_rs <= observed_r)
perm_corr_p_value = (extreme_count + 1) / (n_perm + 1)

print("Observed Pearson correlation:", observed_r)
print("Permutation p-value:", perm_corr_p_value)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(perm_rs, bins=40)
plt.axvline(observed_r, linewidth=2, label="Observed correlation")
plt.xlabel("Permuted Pearson correlations")
plt.ylabel("Frequency")
plt.title("Permutation Test: Correlation of Delivery Time and Review Score")
plt.legend()
plt.show()

## Interpretation and next steps

Compare new outputs with the historical results at the top before updating the README. Explain any differences, especially from multiline CSV parsing.

The original run suggested a state-level payment difference and a negative delivery-time/review-score association. These results describe this dataset; they do not establish causation or represent all Brazilian shoppers.

Further work could use verified geographic boundaries, inspect duplicate reviews and repeated customers, separate canceled orders, assess missing delivery dates, quantify effect sizes and uncertainty, and model delivery delays relative to promised dates. Geographic cleaning in this notebook is a separate exploration; it is not applied to the payment or review analysis.
